In [1]:
import os
os.environ["OPENAI_API_KEY"] = '0'

In [2]:
!pip install -q youtube-transcript-api langchain-community langchain-openai langchain \
               faiss-cpu tiktoken python-dotenv # installed required libraries and dependies

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
!pip install langchain_text_splitters

In [4]:
!pip install youtube-transcript-api

In [5]:
from youtube_transcript_api import TranscriptsDisabled
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_3603/1973181449.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [6]:
!pip install youtube_transcript_api

## Step 1a - Indexing (Document Ingestion)

In [7]:
video_id = "VMj-3S1tku0" # only the ID, not full URL of the  youtube video

In [8]:
ytt_api = YouTubeTranscriptApi() # object of yttranscript API
x = ytt_api.fetch(video_id) # fetching YT transcripts using YT video ID

In [9]:
transcript = " ".join(chunk.text for chunk in x) # Removing time stams and other useless data in transcripts (data preprocessing)
print(transcript)

hello my name is andre and i've been training deep neural networks for a bit more than a decade and in this lecture i'd like to show you what neural network training looks like under the hood so in particular we are going to start with a blank jupiter notebook and by the end of this lecture we will define and train in neural net and you'll get to see everything that goes on under the hood and exactly sort of how that works on an intuitive level now specifically what i would like to do is i would like to take you through building of micrograd now micrograd is this library that i released on github about two years ago but at the time i only uploaded the source code and you'd have to go in by yourself and really figure out how it works so in this lecture i will take you through it step by step and kind of comment on all the pieces of it so what is micrograd and why is it interesting good um micrograd is basically an autograd engine autograd is short for automatic gradient and really what 

In [10]:
x #this is the raw transcripts from YT

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='hello my name is andre', start=0.08, duration=2.88), FetchedTranscriptSnippet(text="and i've been training deep neural", start=1.839, duration=3.041), FetchedTranscriptSnippet(text='networks for a bit more than a decade', start=2.96, duration=3.839), FetchedTranscriptSnippet(text="and in this lecture i'd like to show you", start=4.88, duration=3.759), FetchedTranscriptSnippet(text='what neural network training looks like', start=6.799, duration=4.081), FetchedTranscriptSnippet(text='under the hood so in particular we are', start=8.639, duration=3.601), FetchedTranscriptSnippet(text='going to start with a blank jupiter', start=10.88, duration=3.44), FetchedTranscriptSnippet(text='notebook and by the end of this lecture', start=12.24, duration=4.4), FetchedTranscriptSnippet(text='we will define and train in neural net', start=14.32, duration=3.84), FetchedTranscriptSnippet(text="and you'll get to see everything that", start=16.64

In [11]:
print(transcript) # and this is the cleaned version of same data that has been fetched using youtubes AIP

hello my name is andre and i've been training deep neural networks for a bit more than a decade and in this lecture i'd like to show you what neural network training looks like under the hood so in particular we are going to start with a blank jupiter notebook and by the end of this lecture we will define and train in neural net and you'll get to see everything that goes on under the hood and exactly sort of how that works on an intuitive level now specifically what i would like to do is i would like to take you through building of micrograd now micrograd is this library that i released on github about two years ago but at the time i only uploaded the source code and you'd have to go in by yourself and really figure out how it works so in this lecture i will take you through it step by step and kind of comment on all the pieces of it so what is micrograd and why is it interesting good um micrograd is basically an autograd engine autograd is short for automatic gradient and really what 

## Step 1b - Indexing (Text Splitting)

In [12]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])
# splitting the fetched data into docs of 1000 characters per doc

In [13]:
print(len(chunks))

154


In [14]:
chunks[0]

Document(metadata={}, page_content="hello my name is andre and i've been training deep neural networks for a bit more than a decade and in this lecture i'd like to show you what neural network training looks like under the hood so in particular we are going to start with a blank jupiter notebook and by the end of this lecture we will define and train in neural net and you'll get to see everything that goes on under the hood and exactly sort of how that works on an intuitive level now specifically what i would like to do is i would like to take you through building of micrograd now micrograd is this library that i released on github about two years ago but at the time i only uploaded the source code and you'd have to go in by yourself and really figure out how it works so in this lecture i will take you through it step by step and kind of comment on all the pieces of it so what is micrograd and why is it interesting good um micrograd is basically an autograd engine autograd is short for

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [15]:
!pip install langchain-huggingface

In [16]:
# this code generates the embeddings of the colleted transcripts and stores  them into "FAISS" vector store

from langchain_huggingface import HuggingFaceEmbeddings

embedings = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')

vector_store = FAISS.from_documents(chunks,embedings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [17]:
vector_store.index_to_docstore_id # displayes the IDs of the stored vectors in vector store

{0: '7756adb8-7490-4fc8-b305-1351177f1307',
 1: '3bb0d4e9-1448-4a85-9347-a0cb3da613e3',
 2: '8589655e-b320-4fc1-ab8d-e6145125b75e',
 3: '90121d5e-54a7-42c0-9230-fbb097949d8a',
 4: '79645a09-7537-4cdb-81ed-7368866e67ab',
 5: 'bf5dd063-0e2e-4970-b39a-15c9d94a7df5',
 6: 'db64e21c-2dd1-4a53-ab1f-3c0fcd5071cd',
 7: 'e92fdb48-f979-428f-b6f8-0f219be322c0',
 8: 'bb25f4c5-e7c6-4f92-a5eb-01991221996f',
 9: 'f37e70d2-878c-44af-834f-31f1b6040e9d',
 10: '34b8b221-a606-4f75-9c49-85814071dd27',
 11: '7e3a2aeb-0c53-46c1-84b8-08dba9d56eaa',
 12: '03762501-9a80-4cb4-a49c-781a14043fb0',
 13: '6893bafc-8870-4437-bdec-c29f8b12871a',
 14: '22eff30d-c2ac-4b37-8ecf-43a20664a316',
 15: '603c0a4c-ea69-4377-b958-8ef6dd18067c',
 16: '4894bba1-f465-4cab-b943-22f619da1385',
 17: '730a47df-f071-4939-8662-6df0c6c8271a',
 18: '665f75b6-dc29-468d-b2b3-a949f4340407',
 19: '046cf7e2-1fde-4f34-90c2-3cc2d4c918d2',
 20: '5f4d0beb-662b-45db-9582-c58e1388fd3d',
 21: 'df734454-e10c-4c82-a712-a230ff5ed3d8',
 22: 'e62a8844-bf81-

In [18]:
vector_store.get_by_ids(['f2edd1ee-def4-4744-afab-0d72a9583d6d']) # displays the chunks data by its ID

[]

## Step 2 - Retrieval

In [19]:
#here we have used the most basic type of retriver where it uses vector store as a retriver and retrives "k" no of most simalar docs from the vector base
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [20]:
retriever.invoke('What is Nural net') # retrived 4 docs from vector store is as shown below, which are 4 most similar chunks woth respect to qurey

[Document(id='dc77fa3c-63b4-42a5-ac0a-65e25ec45cfb', metadata={}, page_content="this is something called learning rate decay so in the beginning you have a high learning rate and as the network sort of stabilizes near the end you bring down the learning rate to get some of the fine details in the end and in the end we see the decision surface of the neural net and we see that it learns to separate out the red and the blue area based on the data points so that's the slightly more complicated example and then we'll demo that hyper ymb that you're free to go over but yeah as of today that is micrograd i also wanted to show you a little bit of real stuff so that you get to see how this is actually implemented in production grade library like by torch uh so in particular i wanted to show i wanted to find and show you the backward pass for 10h in pytorch so here in micrograd we see that the backward password 10h is one minus t square where t is the output of the tanh of x times of that grad 

## Step 3 - Augmentation

In [21]:
# making a prompt template
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [22]:
question          = "is the topic of back propagation discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question) # now this retrival will fetch 4 most similar chunks of data from the vector base

In [23]:
retrieved_docs # displaying the retrived dosc

[Document(id='b655822b-f9dc-4001-bba0-dc8dbbdc8011', metadata={}, page_content="one okay so doing the back propagation manually is obviously ridiculous so we are now going to put an end to this suffering and we're going to see how we can implement uh the backward pass a bit more automatically we're not going to be doing all of it manually out here it's now pretty obvious to us by example how these pluses and times are back property ingredients so let's go up to the value object and we're going to start codifying what we've seen in the examples below so we're going to do this by storing a special cell dot backward and underscore backward and this will be a function which is going to do that little piece of chain rule at each little node that compute that took inputs and produced output uh we're going to store how we are going to chain the the outputs gradient into the inputs gradients so by default this will be a function that uh doesn't do anything so um and you can also see that here 

In [24]:
# this code joins the page content of the retrived docs in a one big para
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"one okay so doing the back propagation manually is obviously ridiculous so we are now going to put an end to this suffering and we're going to see how we can implement uh the backward pass a bit more automatically we're not going to be doing all of it manually out here it's now pretty obvious to us by example how these pluses and times are back property ingredients so let's go up to the value object and we're going to start codifying what we've seen in the examples below so we're going to do this by storing a special cell dot backward and underscore backward and this will be a function which is going to do that little piece of chain rule at each little node that compute that took inputs and produced output uh we're going to store how we are going to chain the the outputs gradient into the inputs gradients so by default this will be a function that uh doesn't do anything so um and you can also see that here in the value in micrograb so with this backward function by default doesn't do\

In [25]:
# invoking prompt
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [26]:
final_prompt # displaying the final prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      one okay so doing the back propagation manually is obviously ridiculous so we are now going to put an end to this suffering and we're going to see how we can implement uh the backward pass a bit more automatically we're not going to be doing all of it manually out here it's now pretty obvious to us by example how these pluses and times are back property ingredients so let's go up to the value object and we're going to start codifying what we've seen in the examples below so we're going to do this by storing a special cell dot backward and underscore backward and this will be a function which is going to do that little piece of chain rule at each little node that compute that took inputs and produced output uh we're going to store how we are going to chain the the outputs gradient into the inputs gr

## Step 4 - Generation

In [27]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace # imporing required libraryes

In [ ]:
# setting up hugging face free chatmodel to use for text generation

import os
# Set the token in environment variables specifically for the hub client
os.environ['HUGGINGFACEHUB_API_TOKEN'] = 'paste API key here'

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token=os.environ['HUGGINGFACEHUB_API_TOKEN'],
    temperature=0.2
)

# Some wrappers need the token explicitly passed if environment sync is slow
model = ChatHuggingFace(llm=llm, huggingfacehub_api_token=os.environ['HUGGINGFACEHUB_API_TOKEN'])

In [29]:
# invoking the model where it gets our custom augmented prompt as input and it will genrate text based on the context given in the prompt or it will just retuen with "i dont know"
answer = model.invoke(final_prompt)
print(answer.content)

Yes, the topic of back propagation is discussed in this video. 

The discussion revolves around implementing back propagation manually and then automating it. The speaker explains how back propagation is a recursive application of the chain rule backwards through the computation graph. They also demonstrate how to manually calculate the gradients using the chain rule and how to use these gradients to update the weights in the network.


## Building a Chain

In [30]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [52]:
def format_docs(retrieved_docs):
  # this code joins the page content of the retrived docs in a one big para
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [53]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [54]:
parallel_chain.invoke('what is nural net')

{'context': "this is something called learning rate decay so in the beginning you have a high learning rate and as the network sort of stabilizes near the end you bring down the learning rate to get some of the fine details in the end and in the end we see the decision surface of the neural net and we see that it learns to separate out the red and the blue area based on the data points so that's the slightly more complicated example and then we'll demo that hyper ymb that you're free to go over but yeah as of today that is micrograd i also wanted to show you a little bit of real stuff so that you get to see how this is actually implemented in production grade library like by torch uh so in particular i wanted to show i wanted to find and show you the backward pass for 10h in pytorch so here in micrograd we see that the backward password 10h is one minus t square where t is the output of the tanh of x times of that grad which is the chain rule so we're looking for something that looks l

In [55]:
parser = StrOutputParser()

In [56]:
main_chain = parallel_chain | prompt | model | parser

In [57]:
main_chain.invoke('summerise')

"The code is implementing backpropagation to calculate gradients in a neural network. \n\nHere's a summary of the key points:\n\n1. The code first builds a topological graph of the network, ordering the nodes in a way that allows for efficient backpropagation.\n2. It then uses this graph to calculate the gradients of the network's output with respect to each of the network's parameters.\n3. The gradients are calculated by recursively calling the `backward` method on each node in the graph, starting from the output node.\n4. The gradients are accumulated using the `+=` operator, which allows the gradients to be added together as they flow backwards through the network.\n5. The code also demonstrates how to use numerical gradients to estimate the derivatives of the network's output with respect to its parameters.\n\nThe goal of this code is to implement backpropagation in a neural network, which is a key component of many machine learning algorithms."